# Path A, P8 — hazard table for Step 3, without look-ahead (Oct 27, 2026)

No model is fitted here. Runs: `load_runs("behavior", roles={"train", "val"})` (bot runs and incomplete runs kept). One row per at-risk bin (b = 0.25 s) of a human seat, from the player's first move to `fraught.round_end`; bot positions still enter Agreement. CC runs go through `lab_log` (tick 0.1 s); lab data as logged. Events: `detect_events` with m = 2.

- **Steady at-risk set:** each move against the player's direction starts at e (its first step) and resolves at r (back in the old direction: probe; m steps: reversal; round end: unresolved). Bins with e < t <= r are dropped (the player is mid-event) and counted.
- **Outcomes:** y_rev = a reversal's first step in [t, t + b); y_probe = a probe's first step in [t, t + b).
- **Features at t,** all from the last row strictly before t (time `t_row`): A; trend A(t−) − A((t − w)−) for w = 1 s, 0.5 s, 2 s and 1 / round speed (before the first row: the first row); |s|; at_end (|k| >= 9); time at the end (reset on leaving the band); time since the first step of the last reversal resolved before t (or since the first move); log1p and min(x, 10) of both time features.
- **Round speed** (units/s, raw rows, `interface.py`): lab = the round's own step size x modal pace; CC = the file's speed; the file's speed also when the round has none.

Function: `hazard.build_hazard_table` (`src/slider/hazard.py`). Saved (not in git): `data/processed/hazard_table.parquet`.

In [1]:
import numpy as np
import pandas as pd

from slider import config
from slider import hazard
from slider import interface
from slider import slider_logs as sl
from slider.labclock import CELLS, lab_log
from slider.reversals import run_events, to_steps

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

runs, table = sl.load_runs("behavior", roles={"train", "val"})
ht, dropped = hazard.build_hazard_table(runs)                       # about 1 min
ht.to_parquet(config.HAZARD_TABLE, index=False)
print(len(ht), "rows,", len(dropped), "dropped mid-event bins,", ht["round_id"].nunique(), "rounds,",
      ht["group_id"].nunique(), "groups")

561106 rows, 35218 dropped mid-event bins, 365 rounds, 60 groups


## Per cell: rows, dropped bins, outcomes

In [2]:
def per_cell(h, d, by=()):
    keys = ["topology", "dataset", *by]
    g = h.groupby(keys, sort=False).agg(rows=("y_rev", "size"), sum_y_rev=("y_rev", "sum"), sum_y_probe=("y_probe", "sum"),
                                        mean_y_rev=("y_rev", "mean"), mean_y_probe=("y_probe", "mean"))
    g.insert(1, "dropped", d.groupby(keys).size().reindex(g.index, fill_value=0))
    return g


per_cell(ht, dropped).reindex(CELLS).round(4)

rows  dropped  sum_y_rev  sum_y_probe  mean_y_rev  mean_y_probe
topology  dataset                                                                   
Hierarchy EL        96334     5057       1604          376      0.0167        0.0039
          CL        87420     5344       1224          287      0.0140        0.0033
          CC        84493     3644       1410          387      0.0167        0.0046
Full      CL       157658    12863       2849          693      0.0181        0.0044
          CC        79231     5084       1912          350      0.0241        0.0044
Star      CC        55970     3226        936          226      0.0167        0.0040

In [3]:
by_split = per_cell(ht, dropped, ["split"]).unstack("split").swaplevel(axis=1)
by_split[[(s, c) for s in ["train", "val"] for c in ["rows", "dropped", "sum_y_rev", "sum_y_probe", "mean_y_rev", "mean_y_probe"]]] \
    .reindex(CELLS).round(4)

split              train                                                          val                                                      
                    rows dropped sum_y_rev sum_y_probe mean_y_rev mean_y_probe   rows dropped sum_y_rev sum_y_probe mean_y_rev mean_y_probe
topology  dataset                                                                                                                          
Hierarchy EL       45365    2233       848         149     0.0187       0.0033  50969    2824       756         227     0.0148       0.0045
          CL       55732    3574       913         209     0.0164       0.0038  31688    1770       311          78     0.0098       0.0025
          CC       59157    2263       860         255     0.0145       0.0043  25336    1381       550         132     0.0217       0.0052
Full      CL       80239    5949      1477         447     0.0184       0.0056  77419    6914      1372         246     0.0177       0.0032
          CC       59752    3845      1395         279     0.0233       0.0047  19479    1239       517          71     0.0265       0.0036
Star      CC       35828    1932       580         152     0.0162       0.0042  20142    1294       356          74     0.0177       0.0037

## Checks on the bot-free complete subset

Reversals and their bins are recomputed here: each reversal's first step and the row where it reaches 2 steps (cumulative |Δk|, separate code from `hazard.resolve_times`). Leak counters: kept bins whose features come from a row at time >= t (`t_row`, and |s| checked against the row at `t_row`); kept bins whose time_since_rev counts from a reversal not resolved before t; anchors that are neither the first move nor a reversal's first step.

In [4]:
sub = ht[~ht["bot_run"] & ~ht["incomplete"]]
sub_dropped = dropped[~dropped["bot_run"] & ~dropped["incomplete"]]
kept = {key: g for key, g in sub.groupby(["round_id", "player"])}
rows = []
for run in runs:
    if run["bot"] or run["end"] == "incomplete":
        continue
    if run["dataset"] == "CC":
        t_lab, X_lab = lab_log(run["t"], run["X"])
        run = {**run, "t": t_lab, "X": X_lab}
    s = sl.sign_correct(run)
    k = to_steps(s)
    ev = run_events(run, 2)
    rev = ev[ev["type"] == "reversal"]
    for j, bins in hazard.at_risk_samples(run, 0.25).groupby("player"):
        times = bins["time"].to_numpy()
        g = kept.get((f"{run['file']}#w{run['wave']}", j), sub.iloc[:0])
        t_kept, anchors = g["t"].to_numpy(), (g["t"] - g["time_since_rev"]).to_numpy()
        r = rev[rev["player"] == j]
        first = r["time"].to_numpy()
        done = np.array([run["t"][x.event_row + int(np.argmax(np.cumsum(np.abs(np.diff(k[:, j]))[x.event_row - 1:]) >= 2))]
                         for x in r.itertuples()], dtype=float)
        bin_of = np.round(times[np.searchsorted(times, first + 1e-6, side="right") - 1], 6)
        in_kept = np.isin(bin_of, np.round(t_kept, 6))
        _, counts = np.unique(bin_of[in_kept], return_counts=True)
        hit = np.abs(anchors[:, None] - first[None, :]) < 1e-6
        s_row = np.abs(s[np.searchsorted(run["t"], g["t_row"].to_numpy() - 1e-9), j])
        rows.append({"topology": sl.label_topology(run["A"]), "dataset": run["dataset"], "reversals": len(first),
                     "in dropped bins": int(np.sum(~in_kept)), "bins with 2+ reversals": int(np.sum(counts >= 2)),
                     "extra reversals in shared bins": int(np.sum(counts - 1)),
                     "leak: row at time >= t": int(np.sum(g["t_row"].to_numpy() >= t_kept - 1e-9)),
                     "leak: abs_s not from t_row": int(np.sum(np.abs(s_row - g["abs_s"].to_numpy()) > 1e-9)),
                     "leak: anchor on an unresolved reversal": int(np.sum(hit & (done[None, :] >= t_kept[:, None] - 1e-6))),
                     "anchor not found": int(np.sum(~hit.any(axis=1) & (np.abs(anchors - times[0]) > 1e-6)))})
audit = pd.DataFrame(rows).groupby(["topology", "dataset"], sort=False).sum().reindex(CELLS)

t9 = pd.read_csv(config.TARGET9)
t9 = t9[(t9["set"] == "target9") & (t9["measure"] == "f_time") & (t9["stat"] == "median")].set_index(["topology", "dataset"])["n"]
t8 = pd.read_csv(config.TARGET8)
t8 = t8[t8["set"] == "b_target8"].set_index(["topology", "dataset"])["reversals"]
check = per_cell(sub, sub_dropped)[["rows", "dropped", "sum_y_rev"]].reindex(CELLS)
check["rows + dropped"] = check["rows"] + check["dropped"]
check["target 9 f_time n"] = t9.reindex(CELLS)
check["rows check"] = check["rows + dropped"] == check["target 9 f_time n"]
check["target 8 (b)"] = t8.reindex(CELLS)
check = check.join(audit)
check["target 8 - extra in shared bins"] = check["target 8 (b)"] - check["extra reversals in shared bins"]
check["y_rev check"] = check["sum_y_rev"] == check["target 8 - extra in shared bins"]
check["target 8 - shared - in dropped bins"] = check["target 8 - extra in shared bins"] - check["in dropped bins"]
check.T

topology                               Hierarchy                  Full          Star
dataset                                       EL     CL     CC      CL     CC     CC
rows                                       96334  87420  55433  157658  50321  34546
dropped                                     5057   5344   2488   12863   3378   1818
sum_y_rev                                   1604   1224    936    2849   1242    621
rows + dropped                            101391  92764  57921  170521  53699  36364
target 9 f_time n                         101391  92764  57921  170521  53699  36364
rows check                                  True   True   True    True   True   True
target 8 (b)                                1610   1229    940    2864   1245    622
reversals                                   1610   1229    940    2864   1245    622
in dropped bins                                6      5      4      14      3      1
bins with 2+ reversals                         0      0      0       1      0      0
extra reversals in shared bins                 0      0      0       1      0      0
leak: row at time >= t                         0      0      0       0      0      0
leak: abs_s not from t_row                     0      0      0       0      0      0
leak: anchor on an unresolved reversal         0      0      0       0      0      0
anchor not found                               0      0      0       0      0      0
target 8 - extra in shared bins             1610   1229    940    2863   1245    622
y_rev check                                False  False  False   False  False  False
target 8 - shared - in dropped bins         1604   1224    936    2849   1242    621

In [5]:
print("rows check passes in all cells:", bool(check["rows check"].all()))
print("y_rev check passes in all cells:", bool(check["y_rev check"].all()))
print("sum(y_rev) = target 8 - shared - in dropped bins in all cells:",
      bool((check["sum_y_rev"] == check["target 8 - shared - in dropped bins"]).all()))
print("reversals recomputed = target 8 (b):", bool((check["reversals"] == check["target 8 (b)"]).all()))
leaks = ["leak: row at time >= t", "leak: abs_s not from t_row", "leak: anchor on an unresolved reversal", "anchor not found"]
print({c: int(check[c].sum()) for c in leaks})

rows check passes in all cells: True
y_rev check passes in all cells: False
sum(y_rev) = target 8 - shared - in dropped bins in all cells: True
reversals recomputed = target 8 (b): True
{'leak: row at time >= t': 0, 'leak: abs_s not from t_row': 0, 'leak: anchor on an unresolved reversal': 0, 'anchor not found': 0}


## Features: quantiles, missing values, correlations

In [6]:
ht[hazard.FEATURES].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(3)

,count,mean,std,min,1%,25%,50%,75%,99%,max
A,561106.0,0.615,0.328,0.00,0.000,0.333,0.650,0.975,1.000,1.000
trend,561106.0,0.014,0.132,-1.00,-0.375,0.000,0.000,0.033,0.500,1.000
trend_w05,561106.0,0.007,0.081,-0.95,-0.233,0.000,0.000,0.014,0.300,1.000
trend_w2,561106.0,0.029,0.195,-1.00,-0.533,-0.007,0.000,0.071,0.700,1.000
trend_scaled,561106.0,0.014,0.112,-0.95,-0.340,0.000,0.000,0.029,0.450,1.000
abs_s,561106.0,0.853,0.286,0.00,0.000,0.900,1.000,1.000,1.000,1.000
at_end,561106.0,0.756,0.429,0.00,0.000,1.000,1.000,1.000,1.000,1.000
time_at_end,561106.0,20.464,36.255,0.00,0.000,0.100,6.950,23.150,192.949,297.410
time_at_end_log1p,561106.0,1.970,1.555,0.00,0.000,0.095,2.073,3.184,5.268,5.698
time_at_end_cap10,561106.0,5.648,4.405,0.00,0.000,0.100,6.950,10.000,10.000,10.000


In [7]:
print("missing values: table", int(ht.isna().sum().sum()), "| dropped bins", int(dropped.isna().sum().sum()))
ht.isna().sum()[ht.isna().sum() > 0]

missing values: table 0 | dropped bins 0


Series([], dtype: int64)

In [8]:
ht[hazard.FEATURES].corr().round(2)

,A,trend,trend_w05,trend_w2,trend_scaled,abs_s,at_end,time_at_end,time_at_end_log1p,time_at_end_cap10,time_since_rev,time_since_rev_log1p,time_since_rev_cap10
A,1.00,0.17,0.10,0.25,0.13,0.43,0.44,0.34,0.51,0.50,0.33,0.46,0.43
trend,0.17,1.00,0.83,0.75,0.66,-0.03,-0.03,-0.06,-0.11,-0.12,-0.07,-0.12,-0.14
trend_w05,0.10,0.83,1.00,0.53,0.67,-0.04,-0.04,-0.05,-0.10,-0.10,-0.05,-0.10,-0.12
trend_w2,0.25,0.75,0.53,1.00,0.54,0.00,0.00,-0.08,-0.13,-0.15,-0.09,-0.14,-0.15
trend_scaled,0.13,0.66,0.67,0.54,1.00,-0.01,-0.02,-0.06,-0.10,-0.10,-0.06,-0.08,-0.07
abs_s,0.43,-0.03,-0.04,0.00,-0.01,1.00,0.89,0.29,0.65,0.66,0.24,0.45,0.49
at_end,0.44,-0.03,-0.04,0.00,-0.02,0.89,1.00,0.32,0.72,0.73,0.26,0.49,0.54
time_at_end,0.34,-0.06,-0.05,-0.08,-0.06,0.29,0.32,1.00,0.75,0.53,0.96,0.75,0.41
time_at_end_log1p,0.51,-0.11,-0.10,-0.13,-0.10,0.65,0.72,0.75,1.00,0.94,0.70,0.87,0.74
time_at_end_cap10,0.50,-0.12,-0.10,-0.15,-0.10,0.66,0.73,0.53,0.94,1.00,0.48,0.78,0.80


## The 3 CL files with whole-second timestamps

Step size from their training-topology rounds only: `load_runs("behavior", roles={"train", "val"}, keep_whole_second=True)` (Ring rounds are test rounds and are never returned).

In [9]:
whole = ["log_Fri Sep 16 161000 EDT 2016.txt", "log_Wed Sep 14 155221 EDT 2016.txt", "log_Wed Sep 14 170705 EDT 2016.txt"]
runs_w, table_w = sl.load_runs("behavior", roles={"train", "val"}, keep_whole_second=True)
runs_w = [run for run in runs_w if run["file"] in whole]
iw = interface.interface_table(runs_w)
iw["topologies"] = [", ".join(sorted({sl.label_topology(r["A"]) for r in runs_w if r["file"] == f})) for f in iw["file"]]
iw[["file", "dataset", "split", "version", "rounds", "rows", "time_decimals", "step_size", "pct_on_grid", "topologies"]]

,file,dataset,split,version,rounds,rows,time_decimals,step_size,pct_on_grid,topologies
0,log_Fri Sep 16 161000 EDT 2016.txt,CL,train,none,5,2798,0,0.02,57.852548,"Full, Hierarchy"
1,log_Wed Sep 14 155221 EDT 2016.txt,CL,train,none,6,3477,0,0.02,56.448498,"Full, Hierarchy"
2,log_Wed Sep 14 170705 EDT 2016.txt,CL,train,none,6,6479,0,0.02,62.697065,"Full, Hierarchy"
